<a href="https://colab.research.google.com/github/cassiecinzori/ECON3916/blob/main/Labs/Lecture13/The_Architecture_of_Dimensionality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The Architecture of Dimensionality
### Cassandra Cinzori

In [12]:
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

### Step 1: Ingestion and Naive Model

In [13]:
df = pd.read_csv('Zillow_ZHVI_2026_Micro.csv')

naive_model = smf.ols('Home_Value ~ Property_Age', data=df).fit()
print(naive_model.summary())
print("\nNaive Age Coefficient:", naive_model.params['Property_Age'])

                            OLS Regression Results                            
Dep. Variable:             Home_Value   R-squared:                       0.040
Model:                            OLS   Adj. R-squared:                  0.039
Method:                 Least Squares   F-statistic:                     41.49
Date:                Thu, 19 Mar 2026   Prob (F-statistic):           1.84e-10
Time:                        13:39:25   Log-Likelihood:                -12778.
No. Observations:                1000   AIC:                         2.556e+04
Df Residuals:                     998   BIC:                         2.557e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     3.497e+05   5345.631     65.412   

### Step 2: The Multivariate Model

In [14]:
multi_model = smf.ols('Home_Value ~ Property_Age + Distance_to_Transit', data=df).fit()
print(multi_model.summary())
print("\nMultivariate Age Coefficient:", multi_model.params['Property_Age'])

                            OLS Regression Results                            
Dep. Variable:             Home_Value   R-squared:                       0.045
Model:                            OLS   Adj. R-squared:                  0.043
Method:                 Least Squares   F-statistic:                     23.34
Date:                Thu, 19 Mar 2026   Prob (F-statistic):           1.24e-10
Time:                        13:39:25   Log-Likelihood:                -12776.
No. Observations:                1000   AIC:                         2.556e+04
Df Residuals:                     997   BIC:                         2.557e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept            3.605e+05   7

### Step 3: FWL Theorem Manual Proof
#### 3a: Partial out distance from Price

In [15]:
res_y_model = smf.ols('Home_Value ~ Distance_to_Transit', data=df).fit()
df['Price_Residuals'] = res_y_model.resid

#### 3b: Partial out distance from Age

In [16]:
res_x_model = smf.ols('Property_Age ~ Distance_to_Transit', data=df).fit()
df['Age_Residuals'] = res_x_model.resid

#### 3c: Regress Residuals on Residuals (-1 removes the intercept for exact mathematical matching)

In [17]:
fwl_model = smf.ols('Price_Residuals ~ Age_Residuals - 1', data=df).fit()
print("\nFWL Isolated Age Coefficient:", fwl_model.params['Age_Residuals'])


FWL Isolated Age Coefficient: -745.5855870158623


### AI Expansion

In [19]:
import pandas as pd
import statsmodels.formula.api as smf
import plotly.graph_objects as go
import numpy as np

# --- Load data and fit multivariate model ---
df = pd.read_csv('Zillow_ZHVI_2026_Micro.csv')

multi_model = smf.ols('Home_Value ~ Property_Age + Distance_to_Transit', data=df).fit()

# --- Extract coefficients from statsmodels results ---
# .params returns a Series indexed by variable name
intercept   = multi_model.params['Intercept']
coef_age    = multi_model.params['Property_Age']
coef_dist   = multi_model.params['Distance_to_Transit']

# --- Build a meshgrid to define the regression surface ---
# np.linspace creates evenly spaced values across the observed range of each variable,
# giving us a grid of (age, distance) combinations to evaluate the plane over.
age_range  = np.linspace(df['Property_Age'].min(),       df['Property_Age'].max(),       50)
dist_range = np.linspace(df['Distance_to_Transit'].min(), df['Distance_to_Transit'].max(), 50)

# np.meshgrid turns the two 1D arrays into 2D grids so every (age, dist) pair is covered.
age_grid, dist_grid = np.meshgrid(age_range, dist_range)

# Apply the OLS equation: ŷ = intercept + β₁·age + β₂·distance
# This produces a 2D surface of predicted Home_Value across the grid.
price_surface = intercept + coef_age * age_grid + coef_dist * dist_grid

# --- Build the Plotly figure ---
fig = go.Figure()

# Scatter: actual data points
fig.add_trace(go.Scatter3d(
    x=df['Property_Age'],
    y=df['Distance_to_Transit'],
    z=df['Home_Value'],
    mode='markers',
    marker=dict(size=3, color=df['Home_Value'], colorscale='Viridis', opacity=0.7),
    name='Observed Homes'
))

# Surface: the fitted regression hyperplane
fig.add_trace(go.Surface(
    x=age_grid,
    y=dist_grid,
    z=price_surface,
    colorscale='RdBu',
    opacity=0.5,
    name='Regression Hyperplane',
    showscale=False
))

fig.update_layout(
    title='3D Hedonic Pricing Model: Home Value ~ Property Age + Distance to Transit',
    scene=dict(
        xaxis_title='Property Age (years)',
        yaxis_title='Distance to Transit (miles)',
        zaxis_title='Home Value ($)'
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()

# --- Interpretation ---
print(f"""
=== Hyperplane Interpretation ===

Intercept:              ${intercept:,.2f}
β_Property_Age:         ${coef_age:,.2f}  per year of age
β_Distance_to_Transit:  ${coef_dist:,.2f}  per mile from transit

The 2D plane slicing through 3D space is the OLS hyperplane — it minimizes
squared vertical distances from every point to the surface simultaneously.

The FWL insight: the slope along the Property_Age axis (β = {coef_age:,.2f})
is exactly what the residual-on-residual regression produced. The hyperplane
is holding Distance_to_Transit *constant* as it tilts along the age dimension,
which is geometrically identical to partialling out the confounder first.
A naive 2D regression would have collapsed this plane into a single line,
conflating both slopes — that's OVB.
""")


=== Hyperplane Interpretation ===

Intercept:              $360,468.69
β_Property_Age:         $-745.59  per year of age
β_Distance_to_Transit:  $-1,429.73  per mile from transit

The 2D plane slicing through 3D space is the OLS hyperplane — it minimizes
squared vertical distances from every point to the surface simultaneously.

The FWL insight: the slope along the Property_Age axis (β = -745.59)
is exactly what the residual-on-residual regression produced. The hyperplane
is holding Distance_to_Transit *constant* as it tilts along the age dimension,
which is geometrically identical to partialling out the confounder first.
A naive 2D regression would have collapsed this plane into a single line,
conflating both slopes — that's OVB.

